<a href="https://colab.research.google.com/github/manjari-varma05/Basic-QnA-system-using-simple-RNN/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
df=pd.read_csv('/100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [3]:
#tokenize
def tokenize(text):
  text=text.lower()
  text=text.replace('?','')
  text=text.replace("'",'')
  return text.split()

In [4]:
print(tokenize('What is the capital of France?	'))

['what', 'is', 'the', 'capital', 'of', 'france']


In [5]:
#creating vocabulary
vocab={'<UNK>':0}
def build_vocab(row):
  tok_ques=tokenize(row['question'])
  tok_ans=tokenize(row['answer'])
  for word in tok_ques+tok_ans:
    if word not in vocab:
      vocab[word]=len(vocab)

In [6]:
df.apply(build_vocab,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [7]:
print(len(vocab))

324


In [8]:
def text_to_token(text):
  text=tokenize(text)
  ind=[]
  for word in text:
    if word in vocab:
      ind.append(vocab[word])
    else:
      ind.append(vocab['<UNK>'])
  return ind

In [9]:
import torch
from torch.utils.data import Dataset,DataLoader


In [10]:
class custds(Dataset):
  def __init__(self,df,vocab):
    self.df=df
    self.vocab=vocab
  def __len__(self):
    return self.df.shape[0]
  def __getitem__(self,idx):
    num_ques=text_to_token(self.df.iloc[idx]['question'])
    num_ans=text_to_token(self.df.iloc[idx]['answer'])
    return torch.tensor(num_ques),torch.tensor(num_ans)

In [11]:
ds=custds(df,vocab)

In [12]:
dl=DataLoader(ds,batch_size=1,shuffle=True)

In [13]:
import torch.nn as nn

In [26]:
class simplernn(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.emb=nn.Embedding(vocab_size,embedding_dim=50)
    self.rnn=nn.RNN(50,64,batch_first=True)
    self.fc=nn.Linear(64,vocab_size)
  def forward(self,x):
    x=self.emb(x)
    x,f=self.rnn(x)
    f=f.squeeze(0)
    op=self.fc(f)
    return op


In [27]:
lr=0.001
eps=20

In [28]:
mod=simplernn(len(vocab))
crit=nn.CrossEntropyLoss()
opt=torch.optim.Adam(mod.parameters(),lr=lr)

In [32]:
for ep in range(eps):

    tot_loss = 0

    for q, a in dl:

        opt.zero_grad()

        op = mod(q)

        l = crit(op, a.squeeze(1))

        l.backward()

        opt.step()

        tot_loss = tot_loss + l.item()

    print("Epoch:", ep + 1, "Loss:", tot_loss)

Epoch: 1 Loss: 10.015854846686125
Epoch: 2 Loss: 8.775489564985037
Epoch: 3 Loss: 7.718874920159578
Epoch: 4 Loss: 6.876849349588156
Epoch: 5 Loss: 6.153558369725943
Epoch: 6 Loss: 5.548317082226276
Epoch: 7 Loss: 5.009420234709978
Epoch: 8 Loss: 4.550476050004363
Epoch: 9 Loss: 4.157239958643913
Epoch: 10 Loss: 3.801114296540618
Epoch: 11 Loss: 3.4931346029043198
Epoch: 12 Loss: 3.2136652190238237
Epoch: 13 Loss: 2.9678761959075928
Epoch: 14 Loss: 2.7416698690503836
Epoch: 15 Loss: 2.5380013212561607
Epoch: 16 Loss: 2.355780086480081
Epoch: 17 Loss: 2.196500927209854
Epoch: 18 Loss: 2.042837731540203
Epoch: 19 Loss: 1.9088013209402561
Epoch: 20 Loss: 1.7831267593428493


In [33]:
def predict(mod,ques,threshold=0.5):
  #convert words to numb
  num_ques=text_to_token(ques)
  #convert to tensor
  ques_tens=torch.tensor(num_ques).unsqueeze(0)
  op=mod(ques_tens)
  #logit to prob
  probs=torch.nn.functional.softmax(op,dim=1)
  val,ind=torch.max(probs,dim=1)
  if val<threshold:
    print("Not exactly sure")
  return list(vocab.keys())[ind]


In [35]:
predict(mod,'What is the capital of France')

'paris'